In [1]:
from typing import List
import numpy as np
import time
import matplotlib.pyplot as plt
import torch

import quairkit as qkit
from quairkit import Circuit, Hamiltonian
from quairkit.database import *
from quairkit.loss import *

from quairkit.qinfo import *
from quairkit import *

import pennylane as qml
from pennylane import numpy as np

In [2]:
# Set plot font
font_family = 'STIXGeneral'
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = font_family

qkit.set_dtype('complex128') # default is 'complex64'
# qkit.set_seed(24)  # set a seed for reproducibility

In [ ]:
def f(x):
    """
    f(x) = (-x) * 1_{x < 0}
    """
    return (torch.abs(x) - x )/2


def cheb_coefficients_torch(f, K, N_quad=5000, device="cpu", dtype=torch.float64):
    """
    Compute Chebyshev coefficients c_0,...,c_K
    using Gauss-Chebyshev quadrature
    """
    theta = torch.linspace(0, torch.pi, N_quad, device=device, dtype=dtype)
    x = torch.cos(theta)
    fx = f(x)

    coeffs = torch.zeros(K + 1, device=device, dtype=dtype)
    for n in range(K + 1):
        integrand = fx * torch.cos(n * theta)
        coeffs[n] = (2.0 / torch.pi) * torch.trapz(integrand, theta)

    coeffs[0] *= 0.5  # c_0 normalization
    return coeffs


def chebyshev_matrix_approx_torch(sigma, coeffs):
    """
    Compute sum_n c_n T_n(sigma)
    via matrix Chebyshev recursion
    """
    d = sigma.shape[0]
    K = coeffs.shape[0] - 1

    T0 = torch.eye(d, device=sigma.device, dtype=sigma.dtype)
    T1 = sigma.clone()

    result = coeffs[0] * T0
    if K >= 1:
        result = result + coeffs[1] * T1

    for n in range(2, K + 1):
        Tn = 2.0 * sigma @ T1 - T0
        result = result + coeffs[n] * Tn
        T0, T1 = T1, Tn

    return result


def f_of_sigma_chebyshev_torch(sigma, K, N_quad=5000):
    """
    Return K-th order Chebyshev approximation of f(sigma)
    """
    # 强制 Hermitian
    sigma = 0.5 * (sigma + sigma.conj().transpose(-1, -2))

    # 计算系数
    coeffs = cheb_coefficients_torch(
        f, K,
        N_quad=N_quad,
        device=sigma.device,
        dtype=sigma.dtype
    )

   
    return chebyshev_matrix_approx_torch(sigma, coeffs)


In [ ]:
def complex_entangled_layer_by_hand(num_qubits, depth):
    cir = Circuit(num_qubits)
    
    for d in range(depth):
        cir.rx([0, 1, 2, 3,4,5])
        cir.depolarizing(0.01, [0, 1, 2, 3,4,5])
        cir.rz([0, 1, 2, 3,4,5])
        cir.depolarizing(0.01, [0, 1, 2, 3,4,5])
        
        cir.cx([0,1])
        cir.generalized_depolarizing(0.02, [0, 1])
        cir.cx([1,2])
        cir.generalized_depolarizing(0.02, [1, 2])
        cir.cx([2,3])
        cir.generalized_depolarizing(0.02, [2, 3])
        cir.cx([3,4])
        cir.generalized_depolarizing(0.02, [3, 4])
        cir.cx([4,5])
        cir.generalized_depolarizing(0.02, [4, 5])
        cir.cx([5,0])
        cir.generalized_depolarizing(0.02, [5, 0])

    return cir


In [7]:
def loss_fcn(cir1, cir2, H, budget, random_state_list):
    r"""Compute the loss function of the quantum neural network.

    Args:
        cir: the input PQC
        H: the Hamiltonian to be minimized

    Returns:
        the loss value

    """
    output1 = cir1().trace([2,3,4,5])
    output2 = cir2().trace([2,3,4,5])

    
    # exp_val = ExpecVal(H)   # the expectation value operator of the Hamiltonian H

    c1 = (budget-1)/2 + 1
    c2 = (budget-1)/2

    ave_state = c1*output1.density_matrix - c2*output2.density_matrix
    # print(ave_state)
    negativity = torch.sum(torch.abs(torch.real(trace(random_state_list.density_matrix @ ave_state))) - torch.real(trace(random_state_list.density_matrix @ ave_state)))

    ave_exp = c1*torch.real(trace(output1.density_matrix @ H)) - c2*torch.real(trace(output2.density_matrix @ H))

    evals, evecs = torch.linalg.eig(c1*output1.density_matrix - c2*output2.density_matrix)
    distance = torch.sum(torch.abs(evals))
    
    # cost_fnc = ave_exp + 20 * negativity
    cost_fnc = ave_exp + 10 * torch.abs(torch.real(trace(f_of_sigma_chebyshev_torch(ave_state/budget, K=100))))

    return  cost_fnc, ave_exp, distance # calculate the expectation value of the output state

In [ ]:
def train_model(num_itr, LR, num_qubits, depth, H, budget):
    r"""Train the QNN to find the ground-state energy of the Hamiltonian H.

    Args:
        num_itr: number of training iterations
        LR: learning rate
        num_qubits: number of qubits in the quantum circuit
        depth: number of training layers in the circuit
        H: the Hamiltonian to be minimized
    """

    cir1 = complex_entangled_layer_by_hand(num_qubits, depth)
    cir2 = complex_entangled_layer_by_hand(num_qubits, depth)
    random_state_list = random_state(2, rank=1, size=5000)

    eigenvalues = torch.linalg.eigvalsh(H)    # calculate the eigenvalues and eigenvectors of the Hamiltonian
    lambda_0 = torch.min(torch.real(eigenvalues))    # take the minimum eigenvalue as the ground state energy

    loss_list, time_list = [], []

    opt = torch.optim.Adam(lr=LR, params=list(cir1.parameters()) + list(cir2.parameters())) # cir is a Circuit type
    # opt = torch.optim.SGD(lr=LR, params=list(cir1.parameters()) + list(cir2.parameters())) # cir is a Circuit type
    # opt = torch.optim.AdamW(lr=LR, params=list(cir1.parameters()) + list(cir2.parameters())) # cir is a Circuit type
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', factor=0.5) # activate scheduler

    for itr in range(num_itr):
        start_time = time.time()
        opt.zero_grad()

        loss, ave_exp, distance = loss_fcn(cir1, cir2, H, budget, random_state_list) # compute loss

        loss.backward()
        opt.step()

        loss = loss.item()
        scheduler.step(loss) # activate scheduler

        loss_list.append(loss)
        time_list.append(time.time() - start_time)

        if itr % (num_itr // 10) == 0 or itr == num_itr - 1:
            print(f"iter: {str(itr).zfill(len(str(num_itr)))}, " +
                f"loss: {loss:.8f}, " + f"ave_exp: {ave_exp:.8f}, "+ f"distance: {distance:.8f}, " + 
                f"lr: {scheduler.get_last_lr()[0]:.2E}, avg_time: {np.mean(time_list):.4f}s")
            time_list = []

    display_results(loss_list, lambda_0, cir1)
    return ave_exp
    

In [ ]:
def display_results(loss_list: List[float], lambda_0: int, circuit: Circuit) -> None:
    r"""Plot the training process and the final circuit.

    Args:
        loss_list: the list of loss values during the training process
        lambda_0: the ground-state energy
        circuit: the final circuit

    """

    print("\n" + "-" * 100 + "\n")  # a line of '-' for better readability
    print("Circuit after training:")


    num_itr = len(loss_list)
    base_line = lambda_0 * torch.ones(num_itr)
    list_itr = list(range(num_itr))


    print("\n" + "-" * 100 + "\n")  # a line of '-' for better readability
    print("Training process:")


In [10]:
def Ham_gen(distance):
    symbols = ["He", "H"]
    geometry = np.array([[0.00000000, 0.00000000, -distance],
                        [0.00000000, 0.00000000, distance]])

    molecule = qml.qchem.Molecule(symbols, geometry, charge=1)
    H, qubits = qml.qchem.molecular_hamiltonian(molecule)
    generators = qml.symmetry_generators(H)
    paulixops = qml.paulix_ops(generators, qubits)

    n_electrons = 2
    paulix_sector = qml.qchem.optimal_sector(H, generators, n_electrons)

    H_tapered = qml.taper(H, generators, paulixops, paulix_sector)
    H_tapered_coeffs, H_tapered_ops = H_tapered.terms()
    H_tapered = qml.Hamiltonian(np.real(np.array(H_tapered_coeffs)), H_tapered_ops)
    return H_tapered.matrix()

In [11]:
NUM_ITR = 300  # Set the number of optimization iterations
num_qubits = 6
LR = 0.2     # Set the learning rate
depth = 5
budget = 1.5
energy_list = []

# distance_list = [0.4, 0.45, 0.5,0.6,0.7,0.8,0.9,1.0,1.25,1.7]
distance_list = [0.8]
# for i in range(10):
for distance in distance_list:
    H_mat = torch.tensor(Ham_gen(distance))
    eig_val, eigvec = torch.linalg.eig(H_mat)
    print('The minimum eigenvalue is:', np.real(min(eig_val.numpy())))
    # print(hamiltonian.matrix)
    eigval = train_model(NUM_ITR, LR, num_qubits, depth, H_mat, budget)
    energy_list.append(eigval.detach().numpy().item())
    # print(i, min(energy_list))
print(energy_list)

The minimum eigenvalue is: -2.86054203541046
iter: 000, loss: -1.83516384, ave_exp: -1.83802070, distance: 1.00000000, lr: 2.00E-01, avg_time: 0.1983s
iter: 030, loss: -2.57520487, ave_exp: -2.57682710, distance: 1.00000000, lr: 2.00E-01, avg_time: 0.1669s
iter: 060, loss: -2.60366750, ave_exp: -2.60819183, distance: 1.00000000, lr: 2.00E-01, avg_time: 0.1952s
iter: 090, loss: -2.61905506, ave_exp: -2.62109990, distance: 1.00000000, lr: 2.00E-01, avg_time: 0.2288s
iter: 120, loss: -2.62180705, ave_exp: -2.62378959, distance: 1.00000000, lr: 1.00E-01, avg_time: 0.2124s
iter: 150, loss: -2.61978563, ave_exp: -2.62653280, distance: 1.00000000, lr: 5.00E-02, avg_time: 0.1794s
iter: 180, loss: -2.62276117, ave_exp: -2.62304464, distance: 1.00000000, lr: 1.25E-02, avg_time: 0.1762s
iter: 210, loss: -2.62281396, ave_exp: -2.62368614, distance: 1.00000000, lr: 1.25E-02, avg_time: 0.1808s
iter: 240, loss: -2.62304618, ave_exp: -2.62343351, distance: 1.00000000, lr: 1.25E-02, avg_time: 0.1843s
i

In [1]:

# budget = 1
# [-1.618426854915561, -1.7634033851362156, -2.014347764708238, -2.214162753299387, -2.266700082969064, -2.3870297200240334, -2.4092085784986947, -2.43294371675648, -2.471704173527817, -2.5026326518539888]
# budget = 1.5
# [-1.9753961010951515, -2.1454281498133607, -2.2972168028448325, -2.497033735214472, -2.601331040546258, -2.6245671525107563, -2.652771575104592, -2.656521847051947, -2.6830705165136903, -2.6809861430455477]
# budget = 2
# [-2.2452432985147617, -2.4106329123267543, -2.4147057856135747, -2.541464225693688, -2.6836344266702703, -2.670437586132346, -2.712409600172107, -2.7297447167344595, -2.7588220825825918, -2.7252626318753657]
# budget = 2.5
# [ -2.43567108, -2.58629536, -2.66536354,-2.78589293,-2.810794235165451, -2.83268773, -2.8220583673085438, -2.822408140505216, -2.8054479141644793, -2.789995860767154]